In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 01 · Supervised Fine-Tuning — teach it facts (LIVE)

> **LIVE DEMO LAB.** Resources are suffixed `-live`. The fine-tune job is submitted but not
> awaited — pivot to the pre-demo lab to show the finished model while this one trains.

A Acme member just needs the right copay, but the base model has never seen Acme's copays, cutoffs, or policies — so it confidently invents them. Supervised fine-tuning teaches the facts: train `gpt-4o-mini` on Lab 00's live Q&A pairs, ask the base and tuned models the same question, and read the loss curve to confirm it learned. The tuned model answers `$20` — the exact number in the KB — where the base guessed. *That's the difference between a generic chatbot and a Acme agent.*

---
## Step 1 — Configuration & client

In [ ]:
import os, json, time, requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

load_dotenv()

AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_MODEL               = os.environ.get('BASE_MODEL', 'gpt-4o-mini-2024-07-18')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
SUBSCRIPTION_ID          = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP           = os.environ['AZURE_RESOURCE_GROUP']
RESOURCE_NAME            = os.environ['AZURE_RESOURCE_NAME']
TENANT_ID                = os.environ.get('AZURE_TENANT_ID')

SYSTEM_PROMPT = (
    'You are the Acme Health Member Services assistant. Answer member '
    'questions accurately according to Acme Health\'s official policies, '
    'pharmacy procedures, plan benefits, and the My Health Online portal.'
)

_cred = DefaultAzureCredential(interactive_browser_tenant_id=TENANT_ID) if TENANT_ID else DefaultAzureCredential()

client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
    api_version             = AZURE_OPENAI_API_VERSION,
)

print(f'Endpoint    : {AZURE_OPENAI_ENDPOINT}')
print(f'Base model  : {BASE_MODEL}')
print(f'Base deploy : {BASE_DEPLOYMENT}')

---
## Step 2 — BEFORE: ask the base model 3 Acme-specific questions

These three answers are baked into the Acme knowledge base but do not
exist anywhere on the public internet, so the base model **must** hallucinate.

In [ ]:
TEST_QUESTIONS = [
    'What is the late-fill cutoff time at Acme pharmacies?',
    'How much does a 90-day mail-order refill of a Tier 1 generic cost on Acme Health Plus?',
    'How far in advance does Acme notify members about formulary changes?',
]

CORRECT_ANSWERS = [
    '2:00 PM Pacific. Refills after 2 PM are filled the next business day.',
    '$20 (Tier 1 generic, 60-day copay rate for a 90-day mail-order supply — about 33% less than 3 monthly fills).',
    'At least 60 days in advance, by email to affected members.',
]

print('BEFORE Fine-Tuning - Base Model')
print('=' * 75)

base_answers = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    r = client.chat.completions.create(
        model       = BASE_DEPLOYMENT,
        messages    = [
            { 'role': 'system', 'content': SYSTEM_PROMPT },
            { 'role': 'user',   'content': q },
        ],
        temperature = 0.0,
        max_tokens  = 220,
    )
    a = r.choices[0].message.content.strip()
    base_answers.append(a)
    print(f'Q{i}: {q}')
    print(f'A{i}: {a}\n')

---
## Step 3 — Load the training & validation files from Lab 00

In [ ]:
TRAIN_FILE = Path('data/acme_training_live.jsonl')
VALID_FILE = Path('data/acme_validation_live.jsonl')

with open(TRAIN_FILE, 'r', encoding='utf-8-sig') as f:
    train_data = [json.loads(line) for line in f if line.strip()]
with open(VALID_FILE, 'r', encoding='utf-8-sig') as f:
    valid_data = [json.loads(line) for line in f if line.strip()]

print(f'Training   : {len(train_data)} examples from {TRAIN_FILE}')
print(f'Validation : {len(valid_data)} examples from {VALID_FILE}')

---
## Step 4 — Upload files to Azure OpenAI

In [ ]:
print('Uploading training file...')
train_resp = client.files.create(file=open(TRAIN_FILE, 'rb'), purpose='fine-tune')
training_file_id = train_resp.id
print(f'  -> {training_file_id} ({train_resp.status})')

print('Uploading validation file...')
valid_resp = client.files.create(file=open(VALID_FILE, 'rb'), purpose='fine-tune')
validation_file_id = valid_resp.id
print(f'  -> {validation_file_id} ({valid_resp.status})')

---
## Step 5 — Submit the SFT job

Training tiers: `Standard` (regional, ~$1.70/hr) · `GlobalStandard` (cheaper,
globally distributed) · `Developer` (free, preemptible, deleted after 24h).

In [ ]:
print('Submitting SFT job...')
job = client.fine_tuning.jobs.create(
    training_file   = training_file_id,
    validation_file = validation_file_id,
    model           = BASE_MODEL,
    suffix          = 'acme-v1-live',
    seed            = 42,
    extra_body      = { 'trainingType': 'GlobalStandard' },
)
job_id = job.id
print(f'  Job ID : {job_id}')
print(f'  Status : {job.status}')
print(f'  Base   : {job.model}')

---
## Step 6 — Monitor (polls every 60s)

Typical flow: `queued` -> `validating_files` -> `running` -> `succeeded`.
First SFT runs of `gpt-4o-mini` take 20–60 minutes.

In [ ]:
# [LIVE DEMO] Non-blocking monitor.
# This cell does a SINGLE status check and then returns immediately so the
# presenter can pivot to the pre-demo lab while the job cooks server-side.
# Re-run this cell at the end of the demo to confirm completion.
import os, time
from dotenv import load_dotenv

if 'client' not in globals():
    load_dotenv()
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    _cred = DefaultAzureCredential()
    _tp = get_bearer_token_provider(_cred, 'https://cognitiveservices.azure.com/.default')
    client = AzureOpenAI(
        azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
        api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
        azure_ad_token_provider=_tp,
    )

job_id = globals().get('job_id')
if not job_id:
    print('  no live job_id in scope. Run the previous (submit) step first.')
else:
    js = client.fine_tuning.jobs.retrieve(job_id)
    print(f'  job_id: {job_id}')
    print(f'  status: {js.status}')
    if js.status == 'succeeded':
        fine_tuned_model = js.fine_tuned_model
        print(f'  fine-tuned model: {fine_tuned_model}')
    elif js.status in ('failed', 'cancelled'):
        print(f'  job ended unexpectedly: {js.status}')
        print(getattr(js, 'error', None))
    else:
        print('  >>> [live demo] not blocking. Pivot to the pre-demo lab now.')
        print('  >>> Refresh Foundry portal -> Build -> Fine-tuning to show the run.')
        print('  >>> Re-run this cell at end of demo for final status.')


---
## Step 7 — Plot training metrics

Watch for `train_loss` ↓ and `full_valid_loss` ↓ moving together. Divergence
(train still falling but valid rising) = overfitting → use an earlier checkpoint.

In [ ]:
# Resumable: re-import plotting deps if the kernel was restarted.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

final_job = client.fine_tuning.jobs.retrieve(job_id)
if final_job.result_files:
    rid = final_job.result_files[0]
    content = client.files.content(rid).read()
    Path('data/acme_sft_results_live.csv').write_bytes(content)
    df = pd.read_csv('data/acme_sft_results_live.csv')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('Acme SFT Training Metrics', fontweight='bold')

    ax = axes[0]
    if 'train_loss' in df.columns:
        ax.plot(df['step'], df['train_loss'], label='Train Loss', linewidth=1.5)
    if 'valid_loss' in df.columns:
        m = df['valid_loss'].notna()
        ax.plot(df.loc[m, 'step'], df.loc[m, 'valid_loss'], label='Valid Loss', linewidth=1.5, linestyle='--')
    if 'full_valid_loss' in df.columns:
        m = df['full_valid_loss'].notna()
        ax.scatter(df.loc[m, 'step'], df.loc[m, 'full_valid_loss'], label='Full Valid Loss (epoch end)', color='red', zorder=5)
    ax.set_title('Loss')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    if 'train_mean_token_accuracy' in df.columns:
        ax.plot(df['step'], df['train_mean_token_accuracy'], label='Train Token Acc', linewidth=1.5)
    if 'full_valid_mean_token_accuracy' in df.columns:
        m = df['full_valid_mean_token_accuracy'].notna()
        ax.scatter(df.loc[m, 'step'], df.loc[m, 'full_valid_mean_token_accuracy'], label='Full Valid Acc (epoch end)', color='red', zorder=5)
    ax.set_title('Token Accuracy')
    ax.set_xlabel('Step'); ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('data/acme_sft_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No result file yet.')

---
## Step 8 — Deploy the fine-tuned model (ARM control plane)

Fine-tuned models live in your resource for free, but you have to deploy
them to call them. Deployment costs ~$1.70/hour while live.

In [ ]:
# Resumable: rebuild any state the deploy needs if the kernel was restarted.
import os, json, time, requests
from dotenv import load_dotenv
load_dotenv()

try:
    _cred  # type: ignore[name-defined]
except NameError:
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential(
        interactive_browser_tenant_id=os.environ.get('AZURE_TENANT_ID')
    )

SUBSCRIPTION_ID = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP  = os.environ['AZURE_RESOURCE_GROUP']
RESOURCE_NAME   = os.environ['AZURE_RESOURCE_NAME']

FT_DEPLOYMENT_NAME = 'acme-sft-live-deployment'

auth = _cred.get_token('https://management.azure.com/.default').token
deploy_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}'
    f'/deployments/{FT_DEPLOYMENT_NAME}'
)
r = requests.put(
    deploy_url,
    params  = { 'api-version': '2024-10-01' },
    headers = { 'Authorization': f'Bearer {auth}', 'Content-Type': 'application/json' },
    json    = {
        'sku': { 'name': 'GlobalStandard', 'capacity': 1 },
        'properties': { 'model': { 'format': 'OpenAI', 'name': fine_tuned_model, 'version': '1' } },
    },
)
print(f'HTTP {r.status_code}: {r.reason}')
if r.status_code not in (200, 201):
    print(json.dumps(r.json(), indent=2))

In [ ]:
# Poll until the deployment is live
status_url = deploy_url + '?api-version=2024-10-01'
while True:
    r = requests.get(status_url, headers={ 'Authorization': f'Bearer {auth}' })
    state = r.json().get('properties', {}).get('provisioningState', 'Unknown')
    print(f'[{time.strftime("%H:%M:%S")}] {state}')
    if state == 'Succeeded':
        break
    if state in ('Failed', 'Canceled'):
        print(json.dumps(r.json(), indent=2)); break
    time.sleep(20)

---
## Step 9 — AFTER: same 3 questions on the fine-tuned model

This is the demo. The fine-tuned model should return the exact numbers and
policies from `acme_health_kb.md` — `2:00 PM`, `$20`, `60 days`.

In [ ]:
# Resumable: rebuild whatever is missing after a kernel restart.
import os
from dotenv import load_dotenv
load_dotenv()

try:
    client  # type: ignore[name-defined]
except NameError:
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential(
        interactive_browser_tenant_id=os.environ.get('AZURE_TENANT_ID')
    )
    client = AzureOpenAI(
        azure_endpoint          = os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider = lambda: _cred.get_token(
            'https://cognitiveservices.azure.com/.default'
        ).token,
        api_version             = os.environ['AZURE_OPENAI_API_VERSION'],
    )

FT_DEPLOYMENT_NAME = globals().get('FT_DEPLOYMENT_NAME', 'acme-sft-live-deployment')
SYSTEM_PROMPT = globals().get('SYSTEM_PROMPT', (
    'You are the Acme Health Member Services assistant. Answer member '
    "questions accurately according to Acme Health's official policies, "
    'pharmacy procedures, plan benefits, and the My Health Online portal.'
))
TEST_QUESTIONS = globals().get('TEST_QUESTIONS', [
    'What is the late-fill cutoff time at Acme pharmacies?',
    'How much does a 90-day mail-order refill of a Tier 1 generic cost on Acme Health Plus?',
    'How far in advance does Acme notify members about formulary changes?',
])

print('AFTER Fine-Tuning - SFT Model')
print('=' * 75)

ft_answers = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    r = client.chat.completions.create(
        model       = FT_DEPLOYMENT_NAME,
        messages    = [
            { 'role': 'system', 'content': SYSTEM_PROMPT },
            { 'role': 'user',   'content': q },
        ],
        temperature = 0.0,
        max_tokens  = 220,
    )
    a = r.choices[0].message.content.strip()
    ft_answers.append(a)
    print(f'Q{i}: {q}')
    print(f'A{i}: {a}\n')

---
## Step 10 — Side-by-side

In [ ]:
# Resumable: rebuild anything missing so the side-by-side always works.
import os
from dotenv import load_dotenv
load_dotenv()

try:
    client  # type: ignore[name-defined]
except NameError:
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential(
        interactive_browser_tenant_id=os.environ.get('AZURE_TENANT_ID')
    )
    client = AzureOpenAI(
        azure_endpoint          = os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider = lambda: _cred.get_token(
            'https://cognitiveservices.azure.com/.default'
        ).token,
        api_version             = os.environ['AZURE_OPENAI_API_VERSION'],
    )

BASE_DEPLOYMENT    = globals().get('BASE_DEPLOYMENT', os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini'))
FT_DEPLOYMENT_NAME = globals().get('FT_DEPLOYMENT_NAME', 'acme-sft-live-deployment')
SYSTEM_PROMPT      = globals().get('SYSTEM_PROMPT', (
    'You are the Acme Health Member Services assistant. Answer member '
    "questions accurately according to Acme Health's official policies, "
    'pharmacy procedures, plan benefits, and the My Health Online portal.'
))
TEST_QUESTIONS = globals().get('TEST_QUESTIONS', [
    'What is the late-fill cutoff time at Acme pharmacies?',
    'How much does a 90-day mail-order refill of a Tier 1 generic cost on Acme Health Plus?',
    'How far in advance does Acme notify members about formulary changes?',
])
CORRECT_ANSWERS = globals().get('CORRECT_ANSWERS', [
    '2:00 PM Pacific. Refills after 2 PM are filled the next business day.',
    '$20 (Tier 1 generic, 60-day copay rate for a 90-day mail-order supply - about 33% less than 3 monthly fills).',
    'At least 60 days in advance, by email to affected members.',
])

def _ask(deployment, questions):
    out = []
    for q in questions:
        r = client.chat.completions.create(
            model       = deployment,
            messages    = [
                { 'role': 'system', 'content': SYSTEM_PROMPT },
                { 'role': 'user',   'content': q },
            ],
            temperature = 0.0,
            max_tokens  = 220,
        )
        out.append(r.choices[0].message.content.strip())
    return out

try:
    base_answers  # type: ignore[name-defined]
except NameError:
    print(f'Regenerating base_answers via {BASE_DEPLOYMENT}...')
    base_answers = _ask(BASE_DEPLOYMENT, TEST_QUESTIONS)

try:
    ft_answers    # type: ignore[name-defined]
except NameError:
    print(f'Regenerating ft_answers via {FT_DEPLOYMENT_NAME}...')
    ft_answers = _ask(FT_DEPLOYMENT_NAME, TEST_QUESTIONS)

print('Side-by-Side Comparison')
print('=' * 75)
for i, q in enumerate(TEST_QUESTIONS):
    print(f'\nQ{i+1}: {q}')
    print(f'  Base       : {base_answers[i]}')
    print(f'  Fine-tuned : {ft_answers[i]}')
    print(f'  Correct    : {CORRECT_ANSWERS[i]}')
    print('-' * 75)

print('\nKey takeaway: SFT injects domain-specific FACTS the base model has')
print('no way to know - exact copays, cutoff times, and Acme-only policies.')

---
## Step 11 — Cleanup (delete the deployment, keep the model)

**Important:** deployments cost ~$1.70/hour even with zero traffic. The
fine-tuned model weights are stored for free until you delete them.

In [ ]:
# Resumable cleanup. Stops the ~$1.70/hr deployment clock even after a
# kernel restart - rebuilds whatever state it needs from .env + job history.
import os, requests
from dotenv import load_dotenv
load_dotenv()

try:
    _cred  # type: ignore[name-defined]
except NameError:
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential(
        interactive_browser_tenant_id=os.environ.get('AZURE_TENANT_ID')
    )
try:
    client  # type: ignore[name-defined]
except NameError:
    from openai import AzureOpenAI
    client = AzureOpenAI(
        azure_endpoint          = os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider = lambda: _cred.get_token(
            'https://cognitiveservices.azure.com/.default'
        ).token,
        api_version             = os.environ['AZURE_OPENAI_API_VERSION'],
    )

SUBSCRIPTION_ID    = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP     = os.environ['AZURE_RESOURCE_GROUP']
RESOURCE_NAME      = os.environ['AZURE_RESOURCE_NAME']
FT_DEPLOYMENT_NAME = globals().get('FT_DEPLOYMENT_NAME', 'acme-sft-live-deployment')

# Backfill training/validation file ids from the job if missing
try:
    training_file_id    # type: ignore[name-defined]
    validation_file_id  # type: ignore[name-defined]
except NameError:
    jobs = list(client.fine_tuning.jobs.list())
    if jobs:
        training_file_id   = jobs[0].training_file
        validation_file_id = jobs[0].validation_file
    else:
        training_file_id = validation_file_id = None

# 1) Delete uploaded JSONL files (free, just tidiness)
for fid, name in [(training_file_id, 'training'), (validation_file_id, 'validation')]:
    if not fid:
        print(f'No {name} file id - skip')
        continue
    try:
        client.files.delete(fid); print(f'Deleted {name} file: {fid}')
    except Exception as e:
        print(f'Could not delete {name}: {e}')

# 2) Delete the deployment (THIS is what stops the hourly bill)
deploy_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}'
    f'/deployments/{FT_DEPLOYMENT_NAME}'
)
auth = _cred.get_token('https://management.azure.com/.default').token
r = requests.delete(deploy_url, params={ 'api-version': '2024-10-01' },
                    headers={ 'Authorization': f'Bearer {auth}' })
print(f'Deployment delete ({FT_DEPLOYMENT_NAME}): HTTP {r.status_code}')

fine_tuned_model = globals().get('fine_tuned_model', None)
if fine_tuned_model:
    print(f'\nSave this for later runs:')
    print(f'  FINE_TUNED_MODEL = "{fine_tuned_model}"')